# Task 1: Financial News Exploratory Data Analysis

This notebook develops the evidence base for the later technical-indicator and sentiment-correlation work. It covers headline descriptive statistics, publisher concentration, publisher-domain patterns, news-volume time series, publishing-hour behavior, TF-IDF keywords, recurring phrases, and interpretable market themes.

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loading import load_news
from src.eda import (
    add_headline_length,
    headline_length_stats,
    headline_theme_counts,
    news_volume_spikes,
    publication_frequency,
    publisher_coverage_summary,
    publisher_domains,
    publishing_hour_distribution,
    recurring_phrases,
    top_keywords,
    top_publishers,
)
from src.visualization import plot_news_volume, plot_top_publishers, save_current_figure

sns.set_theme(style='whitegrid')
RAW = PROJECT_ROOT / 'data' / 'raw'
FIGURES = PROJECT_ROOT / 'reports' / 'figures'
FIGURES.mkdir(parents=True, exist_ok=True)

## Load Data

Place the FNSPID news dataset at `data/raw/fnspid_news.csv`. Expected columns are `headline`, `url`, `publisher`, `date`, and `stock`.

In [ ]:
NEWS_FILE = RAW / 'fnspid_news.csv'
if not NEWS_FILE.exists():
    raise FileNotFoundError(f'Missing {NEWS_FILE}. Add the FNSPID news CSV before running Task 1 EDA.')

news = load_news(NEWS_FILE)
news.head()

In [ ]:
print(f'Rows: {len(news):,}')
print(f'Stocks covered: {news["stock"].nunique():,}')
print(f'Publishers: {news["publisher"].nunique():,}')
print(f'Date range: {news["date"].min()} to {news["date"].max()}')
news.info()

## Descriptive Statistics: Headline Length

Headline length helps reveal whether the dataset is made of short alerts, longer editorial headlines, or a mix. This matters because very short headlines may carry less sentiment context while longer headlines may contain richer event information.

In [ ]:
news_len = add_headline_length(news)
headline_summary = headline_length_stats(news).to_frame('headline_characters')
headline_summary

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(news_len['headline_length'], bins=40, kde=True, ax=ax)
ax.set_title('Headline Character Count Distribution')
ax.set_xlabel('Characters')
ax.set_ylabel('Article count')
save_current_figure(FIGURES / 'headline_length_distribution.png')
plt.show()

In [ ]:
news_len.nlargest(10, 'headline_length')[['date', 'stock', 'publisher', 'headline_length', 'headline']]

## Publisher Analysis

This section identifies the most active publishers and whether publisher names include email domains. Heavy publisher concentration can influence the tone and topic mix of the sentiment dataset.

In [ ]:
publisher_counts = top_publishers(news, 15)
publisher_counts.to_frame('article_count')

In [ ]:
fig, ax = plot_top_publishers(publisher_counts)
save_current_figure(FIGURES / 'top_publishers.png')
plt.show()

In [ ]:
publisher_coverage_summary(news, 10)

In [ ]:
publisher_domains(news).head(20).to_frame('article_count')

## Publication Timing and Annotated Volume Spikes

Daily news volume shows whether coverage is steady or clustered around high-attention periods. Spikes are flagged with z-scores so unusually active periods can be reviewed against known earnings, macro, regulatory, or company-specific events.

In [ ]:
daily_volume = publication_frequency(news, 'D')
spikes = news_volume_spikes(news, 'D', z_threshold=2.0)
spikes.head(15)

In [ ]:
fig, ax = plot_news_volume(daily_volume)
for _, row in spikes.head(8).iterrows():
    ax.annotate(
        str(pd.to_datetime(row['period']).date()),
        xy=(row['period'], row['article_count']),
        xytext=(0, 8),
        textcoords='offset points',
        ha='center',
        fontsize=8,
        rotation=45,
    )
ax.set_title('Daily News Volume With High-Volume Spikes Annotated')
save_current_figure(FIGURES / 'daily_news_volume_annotated.png')
plt.show()

In [ ]:
hourly = publishing_hour_distribution(news)
fig, ax = plt.subplots(figsize=(9, 4))
hourly.plot(kind='bar', ax=ax)
ax.set_title('Publishing Hour Distribution')
ax.set_xlabel('Hour of day')
ax.set_ylabel('Article count')
save_current_figure(FIGURES / 'publishing_hours.png')
plt.show()
hourly.to_frame('article_count').T

## Keyword, Phrase, and Topic Signals

TF-IDF highlights distinctive words or short phrases, while phrase counts show repeated headline templates. Theme counts translate keywords into business-readable categories such as earnings, ratings, price movement, corporate actions, and regulatory/legal events.

In [ ]:
keywords = top_keywords(news, 30)
keywords

In [ ]:
phrases = recurring_phrases(news, 30)
phrases

In [ ]:
themes = headline_theme_counts(news)
fig, ax = plt.subplots(figsize=(8, 4))
sns.barplot(data=themes, y='theme', x='article_count', ax=ax, color='steelblue')
ax.set_title('Headline Theme Counts')
ax.set_xlabel('Article count')
ax.set_ylabel('')
save_current_figure(FIGURES / 'headline_theme_counts.png')
plt.show()
themes

## Investment-Relevant EDA Takeaways

After running the notebook, use the tables and annotated plots above to write the strongest observations for the final report:

- **Publisher concentration:** If a few publishers dominate coverage, sentiment signals may partly reflect those publishers' editorial style rather than broad market opinion.
- **Headline length:** Short alert-style headlines may be noisy for sentiment; longer headlines often include richer event context such as earnings, ratings, or regulatory decisions.
- **Volume spikes:** Annotated spikes identify dates where abnormal news attention may coincide with earnings cycles, analyst updates, macro shocks, or company-specific events. These dates are useful candidates for later return and sentiment-event checks.
- **Publishing hour pattern:** Concentration before or near market hours can affect whether sentiment should be aligned to the same trading day or the next trading day.
- **Topics/themes:** Earnings, analyst ratings, price movement, corporate actions, and regulatory/legal themes should be interpreted separately because they may have different price-impact horizons.

These EDA findings connect directly to the investment thesis: news sentiment is more useful when it is filtered by publisher reliability, event theme, publication timing, and confirmation from technical indicators rather than treated as a single raw headline score.